In [ ]:
"""
AgriLST-ML : XGBoost FINAL PRODUCTION MODEL (STATE-WIDE, DISTRICT-CALIBRATED)
=====================================================================
Spatial CV: Leave-One-District-Out (LOGO)
Target: LST_C (Land Surface Temperature in Celsius)
Dataset: state-wide master CSV covering all 36 Maharashtra districts,
5-year rolling historical window.
All 15 numeric + 1 categorical features, spatial LOGO CV, early stopping
per fold, full regularization suite (L1+L2, subsample=0.8,
colsample=0.8, min_child_weight=10), n_estimators derived from mean
early-stop iteration.

TOTAL FEATURES: 15 numeric + 1 categorical (soil_texture) = 16 features
CALIBRATION: + 1 per-district additive bias offset (post-hoc, not a feature)
"""

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
import os

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

# ─────────────────────────────────────────────────────────────────────────────
# 0. PATHS
# ─────────────────────────────────────────────────────────────────────────────
# Point this at the merged output of Data_Preparation_Pipeline v9
# (merge_all_csvs() saves it as agrilst_MASTER_<start_year>_<end_year>.csv,
# now covering all 36 districts / 5-year rolling window — update the filename
# below to match what that run produced).
DATA_PATH  = 'DATA/agrilst_Maharashtra_ALL_2021_2026.csv'
OUTPUT_DIR = 'FINAL_MODEL'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD + FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
print("\nLoading data ...")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"  Loaded: {len(df):,} rows × {len(df.columns)} columns")
print(f"  Districts: {sorted(df['district'].unique())}")

# ── Unit conversions ──────────────────────────────────────────────────────────
df['LST_C']     = df['LST_modis'] - 273.15
df['airtemp_C'] = df['era5_airtemp'] - 273.15

# ── FIX 1: airtemp_C_squared — captures summer LST nonlinearity ───────────────
df['airtemp_C_squared'] = df['airtemp_C'] ** 2

# ── Date / season (evaluation only, not a model feature) ─────────────────────
df['date']   = pd.to_datetime(df['date'])
df['month']  = df['date'].dt.month

def get_season(m):
    if m in [6,7,8,9]:    return 'kharif'
    elif m in [10,11,12]: return 'early_rabi'
    elif m in [1,2,3]:    return 'rabi'
    else:                  return 'summer'

df['season'] = df['month'].apply(get_season)

# ── Spatial macro-proxies ─────────────────────────────────────────────────────
district_elev_map              = df.groupby('district')['elevation'].mean().to_dict()
df['district_mean_elevation']  = df['district'].map(district_elev_map)
df['coastline_distance_proxy'] = df['elevation'] * 0.1 + df['soil_sand'] * 0.05

# ── Interaction features ───────────────────────────────────────────────────────
df['NDVI_x_AirTemp'] = df['ndvi'] * df['airtemp_C']
df['NDWI_div_Clay']  = df['ndwi'] / (df['soil_clay'] + 1.0)

print("Feature engineering complete.")

# ─────────────────────────────────────────────────────────────────────────────
# 2. FEATURE SET DEFINITION
# ─────────────────────────────────────────────────────────────────────────────

# Numeric features (15 total)
NUMERIC_FEATURES = [
    'ndvi',
    'ndwi',
    'airtemp_C',
    'airtemp_C_squared',
    'doy_sin',
    'doy_cos',
    'elevation',
    'district_mean_elevation',
    'soil_clay',
    'soil_sand',
    'rain_0d',
    'rain_15d',
    'coastline_distance_proxy',
    'NDVI_x_AirTemp',
    'NDWI_div_Clay',
]
# Categorical feature
CATEGORICAL_FEATURES = ['soil_texture']

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'LST_C'

# Prepare feature matrix
X = df[ALL_FEATURES].copy()
X['soil_texture'] = X['soil_texture'].astype(int).astype('category')

# Save the full sorted category list from training data
SOIL_TEXTURE_CATEGORIES = sorted(X['soil_texture'].cat.categories.tolist())
print(f"soil_texture categories in training data: {SOIL_TEXTURE_CATEGORIES}")

y      = df[TARGET].copy()
groups = df['district'].copy()

print(f"\nFeature matrix:  {X.shape}")
print(f"  Numeric:     {len(NUMERIC_FEATURES)} features")
print(f"  Categorical: {CATEGORICAL_FEATURES}")
print(f"Target range:    {y.min():.1f}°C – {y.max():.1f}°C  (mean={y.mean():.1f}°C)")

# ─────────────────────────────────────────────────────────────────────────────
# 3. XGBOOST HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

try:
    _test = xgb.XGBRegressor(device='cuda', n_estimators=1, tree_method='hist')
    _test.fit(X.head(100), y.head(100))
    DEVICE = 'cuda'
    print("GPU (CUDA) detected — using GPU for training.")
except Exception:
    DEVICE = 'cpu'
    print("No GPU — using CPU for training.")

XGB_PARAMS = {
    'n_estimators'       : 800,
    'learning_rate'      : 0.05,
    'max_depth'          : 7,
    'min_child_weight'   : 10,
    'subsample'          : 0.8,
    'colsample_bytree'   : 0.8,
    'colsample_bylevel'  : 0.8,
    'reg_alpha'          : 0.1,
    'reg_lambda'         : 1.0,
    'tree_method'        : 'hist',
    'device'             : DEVICE,
    'enable_categorical' : True,
    'random_state'       : 42,
    'n_jobs'             : -1,
    'objective'          : 'reg:squarederror',
    'eval_metric'        : 'rmse',
}

# ─────────────────────────────────────────────────────────────────────────────
# 4. SPATIAL LOGO CROSS-VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("SPATIAL CROSS-VALIDATION — Leave-One-District-Out")
print("="*65)

logo        = LeaveOneGroupOut()
oof_preds   = np.zeros(len(y))
fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups)):
    held_out = groups.iloc[test_idx[0]]

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # 10% internal validation for early stopping
    val_size      = int(len(X_train) * 0.1)
    X_tr,  X_val  = X_train.iloc[:-val_size], X_train.iloc[-val_size:]
    y_tr,  y_val  = y_train.iloc[:-val_size], y_train.iloc[-val_size:]

    model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=30, verbosity=0)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    raw_preds = model.predict(X_test)
    oof_preds[test_idx] = raw_preds

    fold_rmse = np.sqrt(mean_squared_error(y_test, raw_preds))
    fold_mae  = mean_absolute_error(y_test, raw_preds)
    fold_r2   = r2_score(y_test, raw_preds)
    n_trees   = model.best_iteration or XGB_PARAMS['n_estimators']
    fold_bias = float(np.mean(raw_preds - y_test))

    fold_results.append({
        'district' : held_out,
        'rmse_C'   : fold_rmse,
        'mae_C'    : fold_mae,
        'r2'       : fold_r2,
        'bias_C'   : fold_bias,
        'n_rows'   : len(y_test),
        'best_iter': n_trees,
    })
    print(f"  Fold {fold_idx+1:2d} | Held-out: {held_out:<12} | "
          f"RMSE={fold_rmse:.3f}°C  MAE={fold_mae:.3f}°C  R²={fold_r2:.4f}  ")

fold_df = pd.DataFrame(fold_results)

# ── Global OOF Metrics ──────────────────────────────────────────────────────
raw_rmse = np.sqrt(mean_squared_error(y, oof_preds))
raw_mae  = mean_absolute_error(y, oof_preds)
raw_r2   = r2_score(y, oof_preds)
raw_bias = float(np.mean(oof_preds - y))

print(f"\n{'─'*65}")
print(f"GLOBAL OOF (Generalized State-Wide) │  "
      f"RMSE={raw_rmse:.4f}°C  MAE={raw_mae:.4f}°C  R²={raw_r2:.4f}  Bias={raw_bias:+.4f}°C")


# ─────────────────────────────────────────────────────────────────────────────
# 6. PERFORMANCE BREAKDOWN
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("PERFORMANCE BREAKDOWN — GENERALIZED PREDICTIONS")
print("="*65)

oof_df             = df[['district','season','LST_C']].copy()
oof_df['pred']     = oof_preds
oof_df['error']    = oof_df['pred'] - oof_df['LST_C']

# Per-district
print("\n── Per-District ──────────────────────────────────────────────────────")
dist_metrics = []
for district in sorted(df['district'].unique()):
    sub = oof_df[oof_df['district'] == district]
    dist_metrics.append({
        'district': district,
        'rmse_C'  : np.sqrt(mean_squared_error(sub['LST_C'], sub['pred'])),
        'mae_C'   : mean_absolute_error(sub['LST_C'], sub['pred']),
        'r2'      : r2_score(sub['LST_C'], sub['pred']),
        'bias_C'  : float(sub['error'].mean()),
        'n_rows'  : len(sub),
    })
dist_df = pd.DataFrame(dist_metrics)
print(dist_df[['district','rmse_C','mae_C','r2','bias_C','n_rows']].to_string(index=False))

# per-district additive bias correction ──────────────────
# correction[d] = mean(observed − predicted) over this district's OOF rows,
# i.e. exactly -bias_C. Adding it back to a raw prediction re-centers that
# district's predictions on zero mean OOF bias. Computed from the same LOGO
# folds as the reported CV metrics, so it reflects genuine held-out error,
# not in-sample fit.
DISTRICT_BIAS_CORRECTION = {
    row['district']: float(-row['bias_C']) for _, row in dist_df.iterrows()
}
print("\n── Per-District Bias Correction (added to raw prediction) ────────────")
for d, c in DISTRICT_BIAS_CORRECTION.items():
    print(f"  {d:<18}  {c:+.3f}°C")

# Per-season
print("\n── Per-Season ────────────────────────────────────────────────────────")
season_stats = []
for season in ['kharif','early_rabi','rabi','summer']:
    sub = oof_df[oof_df['season'] == season]
    if len(sub) == 0: continue
    season_stats.append({
        'season' : season,
        'n_rows' : len(sub),
        'rmse_C' : np.sqrt(mean_squared_error(sub['LST_C'], sub['pred'])),
        'mae_C'  : mean_absolute_error(sub['LST_C'], sub['pred']),
        'r2'     : r2_score(sub['LST_C'], sub['pred']),
        'bias_C' : float(sub['error'].mean()),
    })
season_df = pd.DataFrame(season_stats)
print(season_df.to_string(index=False))

# District × Season RMSE pivot
print("\n── Per-District × Season RMSE (°C) ───────────────────────────────────")
pivot = oof_df.pivot_table(
    values='error', index='district', columns='season',
    aggfunc=lambda x: np.sqrt(np.mean(x**2))
).round(3)
print(pivot.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 7. RESIDUAL ANALYSIS PLOTS → saved to OUTPUT_DIR
# ─────────────────────────────────────────────────────────────────────────────
print("\nGenerating residual analysis plots ...")

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)
residuals = oof_preds - y.values

# (a) Predicted vs Observed
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y, oof_preds, alpha=0.04, s=1, color='steelblue', rasterized=True)
lims = [y.min(), y.max()]
ax1.plot(lims, lims, 'r--', lw=1.5, label='1:1 line')
ax1.set_xlabel('Observed LST (°C)'); ax1.set_ylabel('Predicted LST (°C)')
ax1.set_title(f'Predicted vs Observed\nRMSE={raw_rmse:.2f}°C  R²={raw_r2:.3f}')
ax1.legend(fontsize=8)

# (b) Residual distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(residuals, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax2.axvline(0, color='red', lw=1.5, linestyle='--')
ax2.axvline(residuals.mean(), color='orange', lw=1.5,
            label=f'Mean={residuals.mean():.3f}°C')
ax2.set_xlabel('Residual (°C)'); ax2.set_ylabel('Count')
ax2.set_title('Residual Distribution'); ax2.legend(fontsize=8)

# (c) Residuals vs Predicted
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(oof_preds, residuals, alpha=0.03, s=1, color='steelblue', rasterized=True)
ax3.axhline(0, color='red', lw=1.5, linestyle='--')
ax3.set_xlabel('Predicted LST (°C)'); ax3.set_ylabel('Residual (°C)')
ax3.set_title('Residuals vs Predicted\n(Heteroscedasticity check)')

# (d) Per-District RMSE
ax4 = fig.add_subplot(gs[1, 0])
x_pos = np.arange(len(dist_df))
dist_rmse_per_dist = dist_df.set_index('district')['rmse_C']
districts_ordered = dist_df['district'].values
ax4.bar(x_pos, [dist_rmse_per_dist[d] for d in districts_ordered], 0.6, color='#4575b4', alpha=0.9)
ax4.axhline(raw_rmse, color='black', lw=1.5, linestyle='--', label=f'Global={raw_rmse:.2f}°C')
ax4.set_xticks(x_pos); ax4.set_xticklabels(districts_ordered, rotation=45, ha='right')
ax4.set_ylabel('RMSE (°C)'); ax4.set_title('Per-District RMSE')
ax4.legend(fontsize=8)

# (e) Per-Season RMSE
ax5 = fig.add_subplot(gs[1, 1])
season_colors = {'kharif':'#4575b4','early_rabi':'#91cf60','rabi':'#fee090','summer':'#d73027'}
for _, row in season_df.iterrows():
    ax5.bar(row['season'], row['rmse_C'], color=season_colors.get(row['season'],'gray'))
ax5.axhline(raw_rmse, color='black', lw=1.5, linestyle='--',
            label=f'Global={raw_rmse:.2f}°C')
ax5.set_ylabel('RMSE (°C)'); ax5.set_title('Per-Season RMSE')
ax5.legend(fontsize=8)

# (f) Bias distribution per district
ax6 = fig.add_subplot(gs[1, 2])
dist_biases = [dist_df[dist_df['district']==d]['bias_C'].values[0] for d in districts_ordered]
ax6.bar(x_pos, dist_biases, 0.6, color='#d73027', alpha=0.9)
ax6.axhline(0, color='black', lw=1, linestyle='--')
ax6.set_xticks(x_pos); ax6.set_xticklabels(districts_ordered, rotation=45, ha='right')
ax6.set_ylabel('Mean Bias (°C)'); ax6.set_title('Per-District Systematic Bias')

# (g) Residuals vs NDVI
ax7 = fig.add_subplot(gs[2, 0])
ax7.scatter(df['ndvi'], residuals, alpha=0.03, s=1, color='forestgreen', rasterized=True)
ax7.axhline(0, color='red', lw=1.5, linestyle='--')
ax7.set_xlabel('NDVI'); ax7.set_ylabel('Residual (°C)')
ax7.set_title('Residuals vs NDVI')

# (h) Residuals vs airtemp_C (summer check)
ax8 = fig.add_subplot(gs[2, 1])
sc = ax8.scatter(df['airtemp_C'], residuals, c=df['airtemp_C_squared'],
                 cmap='RdYlBu_r', alpha=0.03, s=1, rasterized=True)
ax8.axhline(0, color='red', lw=1.5, linestyle='--')
ax8.set_xlabel('Air Temp (°C)'); ax8.set_ylabel('Residual (°C)')
ax8.set_title('Residuals vs Air Temp\n(color=airtemp_C²)')

# (i) Residuals vs rain_15d
ax9 = fig.add_subplot(gs[2, 2])
ax9.scatter(df['rain_15d'], residuals, alpha=0.03, s=1, color='darkorange', rasterized=True)
ax9.axhline(0, color='red', lw=1.5, linestyle='--')
ax9.set_xlabel('rain_15d (mm)'); ax9.set_ylabel('Residual (°C)')
ax9.set_title('Residuals vs 15-day Rainfall')

plt.suptitle(f'AgriLST-ML Generalized Final Model — Spatial LOGO CV Residuals\n'
             f'(RMSE={raw_rmse:.2f}°C  R²={raw_r2:.3f})',
             fontsize=14, y=1.01)
plt.savefig(f'{OUTPUT_DIR}/01_residual_analysis.png', bbox_inches='tight', dpi=120)
plt.show()
print(f"Saved: {OUTPUT_DIR}/01_residual_analysis.png")

# ─────────────────────────────────────────────────────────────────────────────
# 8. FEATURE IMPORTANCE → saved to OUTPUT_DIR
# ─────────────────────────────────────────────────────────────────────────────
print("\nGenerating feature importance plots ...")

last_model = model   # last fold model used as representative

raw_gain   = last_model.get_booster().get_score(importance_type='gain')
raw_weight = last_model.get_booster().get_score(importance_type='weight')

gain_df = pd.DataFrame({
    'feature': ALL_FEATURES,
    'gain'   : [raw_gain.get(f, 0)   for f in ALL_FEATURES],
    'weight' : [raw_weight.get(f, 0) for f in ALL_FEATURES],
}).sort_values('gain', ascending=False)

print("\nFeature Importance (Gain):")
print(gain_df[['feature','gain','weight']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

gi = gain_df.sort_values('gain')
colors_gain = ['#d73027' if g > gain_df['gain'].mean() else '#4575b4'
               for g in gi['gain']]
axes[0].barh(gi['feature'], gi['gain'], color=colors_gain)
axes[0].axvline(gain_df['gain'].mean(), color='black', lw=1.5, linestyle='--',
                label='Mean gain')
axes[0].set_xlabel('Gain'); axes[0].set_title('Feature Importance — Gain')
axes[0].legend(fontsize=9)

wi = gain_df.sort_values('weight')
axes[1].barh(wi['feature'], wi['weight'], color='forestgreen', alpha=0.8)
axes[1].set_xlabel('Weight (splits)'); axes[1].set_title('Feature Importance — Weight')

plt.suptitle('AgriLST-ML Generalized Final Model Feature Importance', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_feature_importance.png', bbox_inches='tight', dpi=120)
plt.show()
print(f"Saved: {OUTPUT_DIR}/02_feature_importance.png")

# ─────────────────────────────────────────────────────────────────────────────
# 9. SUMMER ANALYSIS PLOT → saved to OUTPUT_DIR
# ─────────────────────────────────────────────────────────────────────────────
print("\nGenerating summer analysis plot ...")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Summer vs other RMSE by airtemp_C bucket
bins = np.arange(df['airtemp_C'].min(), df['airtemp_C'].max() + 2, 2)
df['airtemp_bin'] = pd.cut(df['airtemp_C'], bins=bins)
oof_df_ext = oof_df.copy()
oof_df_ext['airtemp_C'] = df['airtemp_C'].values
oof_df_ext['airtemp_bin'] = df['airtemp_bin'].values

bin_rmse = oof_df_ext.groupby('airtemp_bin').apply(
    lambda g: np.sqrt(np.mean((g['pred'] - g['LST_C'])**2)) if len(g) > 10 else np.nan
).dropna()
bin_centers = [iv.mid for iv in bin_rmse.index]
axes[0].bar(bin_centers, bin_rmse.values, width=1.8, color='#d73027', alpha=0.8)
axes[0].axhline(raw_rmse, color='black', lw=1.5, linestyle='--', label=f'Global RMSE={raw_rmse:.2f}°C')
axes[0].set_xlabel('Air Temperature (°C)'); axes[0].set_ylabel('RMSE (°C)')
axes[0].set_title('RMSE by Air Temperature Bucket\n(High temp = harder to predict)')
axes[0].legend(fontsize=9)

# airtemp_C_squared vs residual
axes[1].scatter(df['airtemp_C_squared'], residuals, alpha=0.03, s=1,
                color='crimson', rasterized=True)
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('airtemp_C²'); axes[1].set_ylabel('Residual (°C)')
axes[1].set_title('Residuals vs airtemp_C²\n(Ideally: no trend)')

# Season RMSE comparison
season_order = ['kharif', 'early_rabi', 'rabi', 'summer']
season_rmses = [season_df[season_df['season']==s]['rmse_C'].values[0]
                for s in season_order if s in season_df['season'].values]
axes[2].bar(season_order[:len(season_rmses)], season_rmses,
            color=[season_colors[s] for s in season_order[:len(season_rmses)]])
axes[2].axhline(raw_rmse, color='black', lw=1.5, linestyle='--', label=f'Global={raw_rmse:.2f}°C')
axes[2].set_ylabel('RMSE (°C)'); axes[2].set_title('Per-Season RMSE')
axes[2].legend(fontsize=9)

plt.suptitle('AgriLST-ML — Summer & Temperature Extremes Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_summer_analysis.png', bbox_inches='tight', dpi=120)
plt.show()
print(f"Saved: {OUTPUT_DIR}/03_summer_analysis.png")

# ─────────────────────────────────────────────────────────────────────────────
# 10. PRODUCTION MODEL — TRAIN ON FULL DATASET
# ─────────────────────────────────────────────────────────────────────────────
best_n_trees = int(np.mean([r['best_iter'] for r in fold_results]))
print(f"\nProduction model: {best_n_trees} trees "
      f"(mean early-stopping across {len(fold_results)} folds)")

PROD_PARAMS = {k: v for k, v in XGB_PARAMS.items()}
PROD_PARAMS['n_estimators'] = best_n_trees

production_model = xgb.XGBRegressor(**PROD_PARAMS, verbosity=1)
production_model.fit(X, y)
print("Production model trained on full dataset (536,686 rows, 16 features).")

# ── Save model + metadata ─────────────────────────────────────────────────────
model_path = f'{OUTPUT_DIR}/agrilst_final_xgboost.pkl'
meta_path  = f'{OUTPUT_DIR}/agrilst_final_metadata.pkl'

joblib.dump(production_model, model_path)

metadata = {
    'version'                  : 'final_v1_generalized',
    'numeric_features'         : NUMERIC_FEATURES,
    'categorical_features'     : CATEGORICAL_FEATURES,
    'all_features'             : ALL_FEATURES,
    'target'                   : TARGET,
    'n_estimators'             : best_n_trees,
    'xgb_params'               : PROD_PARAMS,
    'training_rows'            : len(df),
    'districts'                : sorted(df['district'].unique().tolist()),
    'district_elev_map'        : district_elev_map,
    'district_bias_correction' : DISTRICT_BIAS_CORRECTION,
    'cv_rmse'                  : raw_rmse,
    'cv_mae'                   : raw_mae,
    'cv_r2'                    : raw_r2,
    'cv_bias'                  : raw_bias,
    'per_district_cv'          : dist_df.to_dict('records'),
    'per_season_cv'            : season_df.to_dict('records'),
    'feature_importance'       : gain_df.to_dict('records'),
    'soil_texture_categories'  : SOIL_TEXTURE_CATEGORIES,
    'train_date'               : pd.Timestamp.now().strftime('%Y-%m-%d'),
    'fixes_applied'            : [
        'airtemp_C_squared added (FIX 1: summer nonlinearity)',
        'soil_texture categorical enable_categorical=True (FIX 2: Kolhapur/soil types)',
        'per-district additive bias correction restored (FIX 3)',
    ],
}
joblib.dump(metadata, meta_path)
print(f"\nSaved model:    {model_path}")
print(f"Saved metadata: {meta_path}")

# ─────────────────────────────────────────────────────────────────────────────
# 11. FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*65)
print(f"\nModel version:        final_v1_generalized")
print(f"Validation:           Leave-One-District-Out Spatial CV (State-Wide)")
print(f"Training rows:        {len(df):,}")
print(f"Features:             {len(ALL_FEATURES)} (15 numeric + 1 categorical)")
print(f"Best n_estimators:    {best_n_trees}")
print(f"\n── General Performance Metrics ────────────────────────────────────")
print(f"  RMSE: {raw_rmse:.4f}°C")
print(f"  MAE:  {raw_mae:.4f}°C")
print(f"  R²:   {raw_r2:.4f}")
print(f"  Bias: {raw_bias:+.4f}°C")

print(f"\n── Per-District RMSE ──────────────────────────────────────────────")
for _, row in dist_df.iterrows():
    flag = ' ← weakest' if row['rmse_C'] == dist_df['rmse_C'].max() else (
           ' ← best'    if row['rmse_C'] == dist_df['rmse_C'].min() else '')
    print(f"  {row['district']:<12}  {row['rmse_C']:.3f}°C {flag}")

print(f"\n── Per-Season RMSE ────────────────────────────────────────────────")
for _, row in season_df.iterrows():
    print(f"  {row['season']:<12}  {row['rmse_C']:.3f}°C  (n={row['n_rows']:,})")

print(f"\n── Top 5 Features by Gain ─────────────────────────────────────────")
for _, row in gain_df.head(5).iterrows():
    new_flag = '  ← NEW' if row['feature'] in ['airtemp_C_squared','soil_texture'] else ''
    print(f"  {row['feature']:<28}  gain={row['gain']:.1f}{new_flag}")

# ─────────────────────────────────────────────────────────────────────────────
# 12. INFERENCE FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def predict_lst(
    ndvi, ndwi, era5_airtemp_K, doy_sin, doy_cos,
    elevation, soil_clay, soil_sand, soil_texture,
    rain_0d, rain_15d,
    district=None,
    model=None, meta=None
):
    """
    Predict Land Surface Temperature (°C) for one or many agricultural pixels
    anywhere in Maharashtra.

    Parameters — all scalar or array-like, must match in length:
      ndvi, ndwi         : Sentinel-2 vegetation / surface moisture indices
      era5_airtemp_K     : ERA5 2m air temperature in KELVIN
      doy_sin, doy_cos   : day-of-year cyclical encoding
      elevation          : SRTM elevation (m)
      soil_clay          : SoilGrids clay % (0-100)
      soil_sand          : SoilGrids sand % (0-100)
      soil_texture       : SoilGrids USDA texture class (integer 1-12)
      rain_0d            : same-day CHIRPS rainfall (mm)
      rain_15d           : 15-day CHIRPS accumulation (mm)
      district           : Optional district name string or array. Used for
                           (a) district_mean_elevation lookup and (b) the
                           per-district bias correction (FIX 3). Must match
                           a district name seen during training for either
                           to apply — otherwise falls back to raw elevation
                           and zero correction.
      model, meta        : loaded joblib objects. Auto-loaded if None.

    Returns:
      corrected_preds : np.ndarray — raw prediction + per-district bias
                         correction where available, otherwise raw prediction.
      calibrated_mask : np.ndarray[bool] — True where a district-specific
                         correction was actually applied (False = fell back
                         to the uncorrected raw model output).
    """
    import numpy as np
    import pandas as pd
    import joblib

    if model is None:
        model = joblib.load(f'{OUTPUT_DIR}/agrilst_final_xgboost.pkl')
        meta  = joblib.load(f'{OUTPUT_DIR}/agrilst_final_metadata.pkl')

    airtemp_C = np.atleast_1d(era5_airtemp_K).astype(float) - 273.15

    inp = pd.DataFrame({
        'ndvi'             : np.atleast_1d(ndvi).astype(float),
        'ndwi'             : np.atleast_1d(ndwi).astype(float),
        'airtemp_C'        : airtemp_C,
        'airtemp_C_squared': airtemp_C ** 2,
        'doy_sin'          : np.atleast_1d(doy_sin).astype(float),
        'doy_cos'          : np.atleast_1d(doy_cos).astype(float),
        'elevation'        : np.atleast_1d(elevation).astype(float),
        'soil_clay'        : np.atleast_1d(soil_clay).astype(float),
        'soil_sand'        : np.atleast_1d(soil_sand).astype(float),
        'rain_0d'          : np.atleast_1d(rain_0d).astype(float),
        'rain_15d'         : np.atleast_1d(rain_15d).astype(float),
    })

    # Reconstruct spatial proxy features
    if district is not None:
        elev_map = meta['district_elev_map']
        inp['district_mean_elevation'] = (
            pd.Series(np.atleast_1d(district))
            .map(elev_map)
            .fillna(inp['elevation'])
            .values
        )
    else:
        inp['district_mean_elevation'] = inp['elevation'].values

    inp['coastline_distance_proxy'] = inp['elevation'] * 0.1 + inp['soil_sand'] * 0.05
    inp['NDVI_x_AirTemp']           = inp['ndvi'] * inp['airtemp_C']
    inp['NDWI_div_Clay']            = inp['ndwi'] / (inp['soil_clay'] + 1.0)

    # ── FIX: use the EXACT same category list as training ────────────────────
    training_categories = meta.get(
        'soil_texture_categories',
        list(range(1, 13))    # fallback: USDA classes 1-12
    )
    inp['soil_texture'] = pd.Categorical(
        np.atleast_1d(soil_texture).astype(int),
        categories=training_categories
    )

    raw_preds = model.predict(inp[meta['all_features']])

    # ── FIX 3 (restored): apply per-district additive bias correction ────────
    bias_map = meta.get('district_bias_correction', {})
    if district is not None:
        district_arr = pd.Series(np.atleast_1d(district))
        correction   = district_arr.map(bias_map).fillna(0.0).values
        calibrated_mask = district_arr.isin(bias_map.keys()).values
    else:
        correction      = np.zeros(len(raw_preds))
        calibrated_mask = np.zeros(len(raw_preds), dtype=bool)

    corrected_preds = raw_preds + correction

    return corrected_preds, calibrated_mask


# ─────────────────────────────────────────────────────────────────────────────
# SECTION D — Replace inference tests (Section 12, after predict_lst definition)
# ─────────────────────────────────────────────────────────────────────────────

# Load the saved model and metadata fresh (tests the full load → predict path)
model_loaded = joblib.load(model_path)
meta_loaded  = joblib.load(meta_path)

print("\n── Inference function tests ────────────────────────────────────────")

# Test 1: Wardha summer (should be hot ~35–45°C)
p1, calibrated1 = predict_lst(
    ndvi=0.25, ndwi=-0.05, era5_airtemp_K=308.0,
    doy_sin=0.97, doy_cos=-0.26,      # ~April peak
    elevation=310, soil_clay=44, soil_sand=26,
    soil_texture=1,                    # Vertisol (clay) — common in Wardha
    rain_0d=0.0, rain_15d=2.0,
    district='Wardha',
    model=model_loaded, meta=meta_loaded
)

cal_note = "district-calibrated" if calibrated1[0] else "raw (no district correction found)"
print(f"Predicted LST (Wardha Summer Check): {p1[0]:.2f}°C  [{cal_note}]")

import pkg_resources
import os

# List of key libraries used in the notebook
required_packages = [
    'numpy',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn', # sklearn
    'xgboost',
    'joblib'
]

def generate_requirements_file(output_dir, packages):
    requirements_path = os.path.join(output_dir, 'requirements.txt')
    with open(requirements_path, 'w') as f:
        for pkg_name in packages:
            try:
                version = pkg_resources.get_distribution(pkg_name).version
                f.write(f"{pkg_name}=={version}\n")
            except pkg_resources.DistributionNotFound:
                print(f"Warning: Package '{pkg_name}' not found.")
    print(f"Generated requirements.txt at: {requirements_path}")

# Assuming OUTPUT_DIR is defined earlier in the notebook
# If not, you might need to define it or pass a specific path.
# Example: OUTPUT_DIR = '/content/drive/MyDrive/TsHARP_ML_Model/FINAL_MODEL'
generate_requirements_file(OUTPUT_DIR, required_packages)

